# 02 — Data Pipeline

This notebook handles the data pipeline for the anomaly detection project. It includes transcoding videos to a uniform format (10fps H.264), building manifests for training/validation/testing, performing quality checks, and extracting sample clips.

In [ ]:
import os, sys, json, hashlib, csv
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# Mount drive and add project source
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/anomaly_detection'
DATA_ROOT = os.path.join(BASE_DIR, 'data/ucf_crime')

# Add project source to path
PROJECT_SRC = os.path.join(BASE_DIR, 'project_src')
sys.path.insert(0, PROJECT_SRC)

## 1. Pre-transcode Videos

In this step, we pre-transcode all raw videos to a lower resolution and frame rate (128p, 10fps, H.264). This standardizes the dataset, greatly speeds up subsequent loading, and makes it possible to fit more videos onto local Colab storage.

In [ ]:
import subprocess

LOCAL_DATA = '/content/local_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

def transcode_video(src_path, dst_path, fps=10, height=128):
    """Transcode video to low-res H.264 at target fps."""
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    cmd = [
        'ffmpeg', '-y', '-i', src_path,
        '-vf', f'scale=-2:{height}',
        '-r', str(fps),
        '-c:v', 'libx264', '-preset', 'fast',
        '-an',  # no audio
        dst_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0

# Scan and transcode all videos
video_extensions = {'.avi', '.mp4', '.mkv'}
videos_to_transcode = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if Path(f).suffix.lower() in video_extensions:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, DATA_ROOT)
            dst = os.path.join(LOCAL_DATA, Path(rel).with_suffix('.mp4'))
            if not os.path.exists(dst):
                videos_to_transcode.append((src, dst))

print(f'Videos to transcode: {len(videos_to_transcode)}')
for src, dst in tqdm(videos_to_transcode, desc='Transcoding'):
    success = transcode_video(src, dst)
    if not success:
        print(f'  FAILED: {src}')

## 2. Build Manifest

Here we scan the transcoded videos, parse their labels from folder or file names according to UCF-Crime structure, and build the full data manifest including the official test split.

In [ ]:
# UCF-Crime official split
# The official test set file lists 290 test videos
# Download from: https://www.dropbox.com/sh/75v5ehq4cdg5g5g/AABvnJSwZI7zXb8_myBA0CLHa?dl=0

# Load official test split if available
official_test_file = os.path.join(DATA_ROOT, 'Temporal_Anomaly_Annotation_for_Testing_Videos.txt')
official_test_videos = set()
if os.path.exists(official_test_file):
    with open(official_test_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                official_test_videos.add(parts[0].split('.')[0])  # video name without extension
    print(f'Official test videos loaded: {len(official_test_videos)}')
else:
    print('WARNING: Official test annotation file not found.')
    print(f'Expected at: {official_test_file}')

# Anomaly categories in UCF-Crime
ANOMALY_CATEGORIES = [
    'Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion',
    'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting',
    'Stealing', 'Vandalism'
]

manifest_rows = []
for root, dirs, files in os.walk(LOCAL_DATA):
    for f in sorted(files):
        if not f.endswith('.mp4'):
            continue
        video_path = os.path.join(root, f)
        video_name = Path(f).stem
        
        # Determine label and category
        category = 'Normal'
        label = 0
        for cat in ANOMALY_CATEGORIES:
            if video_name.startswith(cat) or cat in video_path:
                category = cat
                label = 1
                break
        
        # Determine split using official UCF-Crime split
        if video_name in official_test_videos or any(video_name.startswith(v) for v in official_test_videos):
            split = 'test'
        else:
            split = 'train'  # will carve validation from this later
        
        manifest_rows.append({
            'video_path': video_path,
            'video_name': video_name,
            'label': label,
            'category': category,
            'split': split,
        })

df = pd.DataFrame(manifest_rows)
print(f'Total videos: {len(df)}')
print(f'Normal: {(df.label==0).sum()}, Anomalous: {(df.label==1).sum()}')
print(f'Official test set: {(df.split=="test").sum()}')
print(f'Train set: {(df.split=="train").sum()}')

In [ ]:
def quality_check(video_path, min_frames=16):
    """Returns (is_valid, num_frames, fps, reason)."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return False, 0, 0, 'cannot_open'
    fps = cap.get(cv2.CAP_PROP_FPS)
    num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if num_frames < min_frames:
        return False, num_frames, fps, f'too_few_frames ({num_frames})'
    if fps <= 0:
        return False, num_frames, fps, 'invalid_fps'
    return True, num_frames, fps, 'ok'

valid_mask = []
rejections = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Quality check'):
    is_valid, nf, fps, reason = quality_check(row['video_path'])
    valid_mask.append(is_valid)
    if not is_valid:
        rejections.append({'video': row['video_name'], 'reason': reason, 'frames': nf})

df = df[valid_mask].reset_index(drop=True)
print(f'\nValid videos: {len(df)}')
print(f'Rejected: {len(rejections)}')
if rejections:
    rej_df = pd.DataFrame(rejections)
    print(rej_df.to_string())
    rej_df.to_csv(os.path.join(BASE_DIR, 'data_quality_rejections.csv'), index=False)

In [ ]:
from sklearn.model_selection import train_test_split

train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

# Stratified split: 85% train, 15% validation from the official train set
train_indices, val_indices = train_test_split(
    train_df.index, test_size=0.15, random_state=42, stratify=train_df['label']
)

df.loc[val_indices, 'split'] = 'val'

print('Final split counts:')
print(df['split'].value_counts())
print('\nPer-split label distribution:')
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

In [ ]:
for split_name in ['train', 'val', 'test']:
    split_df = df[df['split'] == split_name]
    out_path = os.path.join(BASE_DIR, f'manifest_{split_name}.csv')
    split_df.to_csv(out_path, index=False)
    print(f'Saved {out_path}: {len(split_df)} videos')

# Also save full manifest
df.to_csv(os.path.join(BASE_DIR, 'manifest_full.csv'), index=False)
print(f'\nFull manifest saved: {len(df)} videos')

In [ ]:
from training.dataset.transforms import VideoTransform, decode_video_clips
import matplotlib.pyplot as plt

sample_videos = df.sample(3, random_state=42)

for _, row in sample_videos.iterrows():
    clips = decode_video_clips(row['video_path'], target_fps=10, clip_length=16, clip_stride=8)
    print(f"\n{row['video_name']} ({row['category']}): {len(clips)} clips")
    if clips:
        clip_frames, start_t, end_t = clips[0]
        print(f'  Clip shape: {clip_frames.shape}, dtype: {clip_frames.dtype}')
        print(f'  Time: {start_t:.2f}s - {end_t:.2f}s')
        
        # Show first frame of first clip
        fig, axes = plt.subplots(1, 4, figsize=(12, 3))
        for i, ax in enumerate(axes):
            frame_idx = i * 4  # show frames 0, 4, 8, 12
            ax.imshow(clip_frames[frame_idx])
            ax.set_title(f'Frame {frame_idx}')
            ax.axis('off')
        plt.suptitle(f"{row['video_name']} - First clip")
        plt.tight_layout()
        plt.show()
        
        # Test transform
        transform = VideoTransform(mode='train')
        tensor = transform(clip_frames)
        print(f'  Transformed tensor shape: {tensor.shape}, dtype: {tensor.dtype}')

## Summary

Data pipeline is now complete. We have successfully transcoded the videos, filtered out invalid ones, separated into splits, and saved CSV manifests to Google Drive. Sample clips are successfully decoded and visualised.